In [ ]:
import sqlite3
import pandas as pd
from openai import OpenAI
from config import API_KEY

# المسارات النسبية للملفات
DB_PATH = "../database/ecommerce.db"
SCHEMA_PATH = "../database/schema.sql"

# قراءة الـ Schema من الملف
with open(SCHEMA_PATH, 'r', encoding='utf-8') as f:
    schema_text = f.read()

def execute_sql(sql_query):
    """دالة لتنفيذ الـ SQL وإرجاع النتائج كـ DataFrame"""
    conn = sqlite3.connect(DB_PATH)
    try:
        # تنظيف الكود في حال رجع الموديل Markdown blocks
        cleaned_sql = sql_query.replace("```sql", "").replace("```", "").strip()
        df = pd.read_sql_query(cleaned_sql, conn)
        conn.close()
        return df
    except Exception as e:
        conn.close()
        return f"❌ Error executing SQL: {e}"

print("✅ Schema loaded & execution function ready!")

✅ Schema loaded & execution function ready!


In [ ]:
# ضع الـ API Key بتاعك من OpenRouter هنا
OPENROUTER_API_KEY = API_KEY
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# صياغة الـ System Prompt المخصص للـ Text-to-SQL
system_prompt = f"""You are a strict text-to-SQL engine.
Given the SQLite database schema below, write a valid SQLite query to answer the user's question.
Rules:
1. Output ONLY the raw SQL query.
2. Do NOT include markdown code blocks, explanations, or quotes.
3. Use correct JOINs and table relationships as defined in the schema.

Database Schema:
{schema_text}
"""

print("✅ OpenRouter Client configured.")

✅ OpenRouter Client configured.


In [3]:
def test_baseline_question(user_question):
    print(f"❓ السؤال: {user_question}")
    
    # 1. إرسال السؤال للموديل (Qwen2.5-Coder المجاني)
    response = client.chat.completions.create(
        model="qwen/qwen-2.5-coder-32b-instruct",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question}
        ],
        temperature=0.0
    )
    
    generated_sql = response.choices[0].message.content.strip()
    print("🤖 الـ SQL المولد:")
    print(generated_sql)
    print("\n📊 النتيجة من الداتابيز:")
    
    # 2. تنفيذ الاستعلام فوراً على ecommerce.db
    result = execute_sql(generated_sql)
    display(result)
    print("=" * 60)

# ----------------------------------------------------
# تجربة أسئلة متدرجة الصعوبة (Baseline Tests)
# ----------------------------------------------------

# تجربة 1: استعلام بسيط من جدول واحد
test_baseline_question("عرض أسامي العملاء وإيميلاتهم اللي من مصر")

# تجربة 2: استعلام مع أرقام وتجميع (Aggregation & JOIN)
test_baseline_question("أوجد إجمالي مبيعات كل عميل واسمه بالكامل مرتبين من الأكثر للشراء")

# تجربة 3: استعلام معقد على أكثر من جدول (Multi-table JOIN)
test_baseline_question("ما هي أسماء المنتجات الأكثر مبيعاً من حيث إجمالي الكمية المباعة؟")

❓ السؤال: عرض أسامي العملاء وإيميلاتهم اللي من مصر
🤖 الـ SQL المولد:
SELECT first_name, last_name, email FROM customers WHERE country = 'مصر'

📊 النتيجة من الداتابيز:


,first_name,last_name,email


❓ السؤال: أوجد إجمالي مبيعات كل عميل واسمه بالكامل مرتبين من الأكثر للشراء
🤖 الـ SQL المولد:
SELECT c.first_name, c.last_name, SUM(o.total_amount) AS total_spent
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC

📊 النتيجة من الداتابيز:


,first_name,last_name,total_spent
0,Ahmed,Ali,195.50
1,Sara,Mohamed,150.00
2,Fatima,Hassan,120.00
3,John,Doe,19.99


❓ السؤال: ما هي أسماء المنتجات الأكثر مبيعاً من حيث إجمالي الكمية المباعة؟
🤖 الـ SQL المولد:
SELECT p.product_name, SUM(oi.quantity) AS total_quantity_sold
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.product_id
ORDER BY total_quantity_sold DESC

📊 النتيجة من الداتابيز:


,product_name,total_quantity_sold
0,Mechanical Gaming Keyboard,2
1,Coffee Maker,1
2,Running Shoes,1
3,Cotton T-Shirt,1
4,Noise Cancelling Headphones,1
5,Wireless Laptop Mouse,1
